# 02 — Training Curves

Visualise fine-tuning convergence: loss curves, per-epoch accuracy and F1.

**Run after:** `make train`

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

state = json.loads((Path("models/classifier") / "checkpoint-777" / "trainer_state.json").read_text())
history = pd.DataFrame(state["log_history"])
eval_log = history.dropna(subset=["eval_loss"])
print(f"Epochs logged: {eval_log['epoch'].max():.0f}")

In [ ]:
best_epoch = eval_log.loc[eval_log["eval_f1_macro"].idxmax()]

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Val loss
axes[0].plot(eval_log["epoch"], eval_log["eval_loss"], "o-", color="#2563eb", linewidth=2, markersize=5)
axes[0].axvline(best_epoch["epoch"], color="#dc2626", linestyle="--", alpha=0.5, label="best checkpoint")
axes[0].set(title="Validation Loss", xlabel="Epoch", ylabel="Loss")
axes[0].legend(fontsize=8)
axes[0].grid(alpha=0.3)

# Val F1 macro
axes[1].plot(eval_log["epoch"], eval_log["eval_f1_macro"], "o-", color="#2563eb", linewidth=2, markersize=5)
axes[1].axvline(best_epoch["epoch"], color="#dc2626", linestyle="--", alpha=0.5, label="best checkpoint")
axes[1].set(title="Validation F1 Macro", xlabel="Epoch", ylabel="F1")
axes[1].legend(fontsize=8)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig("reports/training_curves.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
best = eval_log.sort_values("eval_f1_macro", ascending=False).iloc[0]
print(f"Best epoch:      {best['epoch']:.0f}")
print(f"Val accuracy:    {best['eval_accuracy']:.4f}")
print(f"Val F1 macro:    {best['eval_f1_macro']:.4f}")
print(f"Val F1 weighted: {best['eval_f1_weighted']:.4f}")

## Hyperparameter comparison

Compare key training settings and their effect on final validation metrics. Re-run with different `configs/training.yaml` values and record results below.

In [ ]:
import yaml

config = yaml.safe_load(Path("configs/training.yaml").read_text())

# Current run summary
current = {
    "model":         config["model_name"],
    "lr":            config["learning_rate"],
    "batch_size":    config["batch_size"],
    "epochs_run":    int(eval_log["epoch"].max()),
    "warmup_ratio":  config["warmup_ratio"],
    "weight_decay":  config["weight_decay"],
    "best_f1_macro": float(best["eval_f1_macro"]),
    "best_accuracy": float(best["eval_accuracy"]),
}

# Append to comparison log (add rows manually for different runs)
comparison = pd.DataFrame([current])
print(comparison.to_string(index=False))